# NGH-Seis v1.0 quick start
This notebook reads the frozen manifest, loads one complete sample, checks the marine-streamer geometry, and displays the primary input and targets. Set `PROJECT` to the extracted repository root when using a downloaded release.

In [ ]:
from pathlib import Path
import json
import numpy as np
import matplotlib.pyplot as plt
ROOT = Path.cwd().resolve()
if ROOT.name == 'examples': ROOT = ROOT.parent
PRODUCTION = ROOT / 'outputs' / 'NGH-Seis_v1.0'
RELEASE = PRODUCTION / 'metadata'
manifest = json.loads((RELEASE / 'accepted_manifest.json').read_text(encoding='utf-8'))
len(manifest), manifest[0]['candidate_id'], manifest[0]['style']


In [ ]:
record = manifest[0]
with np.load(PRODUCTION / record['pair'], allow_pickle=False) as pair:
    arrays = {name: pair[name].copy() for name in [
        'input_rtm_conditioned_unscaled', 'input_rtm_display_only',
        'target_Sh', 'target_Sg', 'auxiliary_Vp', 'migration_Vp',
        'loss_mask', 'shot_x_m', 'receiver_x_m']}
with np.load(PRODUCTION / record['observations'], allow_pickle=False) as obs:
    raw_shape = obs['raw_gathers'].shape
    processed_shape = obs['processed_gathers'].shape
print('pair shape:', arrays['target_Sh'].shape)
print('raw/processed full-gather shapes:', raw_shape, processed_shape)
print('effective shot spacing (m):', np.unique(np.diff(arrays['shot_x_m'])))
print('all receivers trail their sources:', bool(np.all(arrays['receiver_x_m'] < arrays['shot_x_m'][:, None])))

In [ ]:
extent = [0.0, 3.525, 1.341, -0.192]
fig, axes = plt.subplots(1, 4, figsize=(15, 5), constrained_layout=True)
panels = [
    ('input_rtm_display_only', 'Adjoint migration input', 'gray'),
    ('auxiliary_Vp', 'True Vp (m/s)', 'turbo'),
    ('target_Sh', 'Hydrate saturation Sh', 'viridis'),
    ('target_Sg', 'Free-gas saturation Sg', 'magma'),
]
for ax, (name, title, cmap) in zip(axes, panels):
    image = ax.imshow(arrays[name], extent=extent, aspect='auto', cmap=cmap)
    ax.set(title=title, xlabel='distance (km)', ylabel='depth relative to seafloor (km)')
    fig.colorbar(image, ax=ax, shrink=0.7)
plt.show()

For quantitative training, use `input_rtm_conditioned_unscaled` with the scale stored in `rtm_amplitude_calibration.json`. `input_rtm_display_only` is only for visualization. `loss_mask` is used for loss and metrics and is not a network input.